# Accessing GeoServer data

This notebooks tests the access to different GeoServer resources and checks that the different user permissions from Magpie are respected.

This notebook expects that the evaluated server instance has the following components properly configured:
- [`components/geoserver`](https://github.com/bird-house/birdhouse-deploy/tree/master/birdhouse/components/geoserver)
- [`optional-components/test-geoserver-secured-access`](https://github.com/bird-house/birdhouse-deploy/tree/master/birdhouse/optional-components/test-geoserver-secured-access)

In [1]:
# define some useful variables for following steps
import copy
import json
import os
import requests
import urllib3
import uuid

print("Setup configuration parameters...")

VERIFY_SSL = True if 'DISABLE_VERIFY_SSL' not in os.environ else False
if not VERIFY_SSL:
    urllib3.disable_warnings()  # disable warnings for using https without certificate verification enabled

HEADERS = {"Accept": "application/json", "Content-Type": "application/json"}

PAVICS_HOST = os.getenv("PAVICS_HOST", "pavics.ouranos.ca")
assert PAVICS_HOST != "", "Invalid PAVICS HOST value."

TEST_DISABLE_CLEANUP = str(os.getenv("TEST_DISABLE_CLEANUP", False)).lower() in ["true", "1", "on"]

TEST_PROXY_SCHEME = os.getenv("TEST_PROXY_SCHEME", "https")
TEST_SERVER_URL = str(os.getenv("TEST_SERVER_URL") or f"{TEST_PROXY_SCHEME}://{PAVICS_HOST}").rstrip("/")

# Magpie variables
MAGPIE_URL = f"{TEST_SERVER_URL}/magpie"

def get_credentials(var_name):
    value = os.getenv(var_name)
    if not value:
        raise ValueError("Missing test admin credentials `{}` to run tests.".format(var_name))
    return value

TEST_MAGPIE_ADMIN_USERNAME = get_credentials("TEST_MAGPIE_ADMIN_USERNAME")
TEST_MAGPIE_ADMIN_PASSWORD = get_credentials("TEST_MAGPIE_ADMIN_PASSWORD")
TEST_GEOSERVER_ADMIN_USERNAME = get_credentials("TEST_GEOSERVER_ADMIN_USERNAME")
TEST_GEOSERVER_ADMIN_PASSWORD = get_credentials("TEST_GEOSERVER_ADMIN_PASSWORD")

MAGPIE_TEST_USER = os.getenv("MAGPIE_TEST_USER") or "test-user-{!s}".format(uuid.uuid4())
MAGPIE_TEST_PASSWORD = os.getenv("MAGPIE_TEST_PASSWORD") or str(uuid.uuid4())
MAGPIE_NO_PERM_USER = os.getenv("MAGPIE_NO_PERM_USER") or "test-user-no-perm-{!s}".format(uuid.uuid4())
MAGPIE_NO_PERM_PASSWORD = os.getenv("MAGPIE_NO_PERM_PASSWORD") or str(uuid.uuid4())

# Geoserver variables
GEOSERVER_SERVICE = os.getenv("GEOSERVER_SERVICE") or "geoserver"
GEOSERVER_URL = f"{TEST_SERVER_URL}/{GEOSERVER_SERVICE}"
GEOSERVER_REST_URL = f"{TEST_SERVER_URL}/geoserver/rest"

workspace_name = "test_workspace"
datastore_name = "test_datastore"
geoserver_datastore_path = "/geoserver-test-data"
shapefile_name = "Espace_Vert"

print("  Will use Magpie URL:   [{}]".format(MAGPIE_URL))
print("  Will use Geoserver URL:   [{}]".format(GEOSERVER_URL))
print("  Will use Geoserver rest URL:  [{}]".format(GEOSERVER_REST_URL))

Setup configuration parameters...
  Will use Magpie URL:   [https://pavics.ouranos.ca/magpie]
  Will use Geoserver URL:   [https://pavics.ouranos.ca/geoserver]
  Will use Geoserver rest URL:  [https://pavics.ouranos.ca/geoserver/rest]


In [4]:
# check that expected components are present
# if not, don't fail to allow flexibility with other setups, but at least make it clear that it might be the cause of failure
components_url = f"{TEST_SERVER_URL}/components"
components_resp = requests.request(url=components_url, headers=HEADERS, method="GET", timeout=10, verify=VERIFY_SSL)
components_src = "bird-house/birdhouse-deploy"
components_req = [
    f"{components_src}:components/geoserver",
    f"{components_src}:optional-components/test-geoserver-secured-access",
]
components = []
if components_resp.status_code != 200:
    print(
        f"⚠️ WARNING: Could not validate components offered by the server. "
        f"Got [{components_resp.status_code}] from [{components_url}]."
    )
else:
    components = components_resp.json().get("components") or []
if not all(comp_req in components for comp_req in components_req):
    print(
        f"⚠️ WARNING: Expected components not found in the server. "
        f"Expected to find:\n{json.dumps(sorted(components_req), indent=2)}\n"
        f"in:\n{json.dumps(sorted(components), indent=2)}"
    )
else:
    print("✅ Expected components found in the server, proceeding with tests.")

✅ Expected components found in the server, proceeding with tests.


In [5]:
def magpie_signin(user_name, password):
    signin_url = f"{MAGPIE_URL}/signin"
    data = {"user_name": user_name, "password": password}
    try:
        resp = requests.request(url=signin_url, headers=HEADERS, method="POST", json=data, timeout=10, verify=VERIFY_SSL)
    except Exception as exc:
        raise RuntimeError(f"Failed to sign in to Magpie (url: `{signin_url}`) with user `{data['user_name']}`. "
                           f"Exception : {exc}. ")
    if resp.status_code != 200:
        raise RuntimeError(f"Unexpected response while trying to sign in to Magpie with user `{user_name}` : {resp.text}")
    return resp

In [6]:
magpie_admin_session = requests.Session()
magpie_admin_session.verify = VERIFY_SSL
magpie_admin_session.headers = HEADERS
magpie_admin_session.cookies = magpie_signin(TEST_MAGPIE_ADMIN_USERNAME, TEST_MAGPIE_ADMIN_PASSWORD).cookies

geoserver_admin_session = copy.deepcopy(magpie_admin_session)
geoserver_admin_session.auth = (TEST_GEOSERVER_ADMIN_USERNAME, TEST_GEOSERVER_ADMIN_PASSWORD)

test_user_session = requests.Session()
test_user_session.verify = VERIFY_SSL
test_user_session.headers = HEADERS

no_perm_user_session = requests.Session()
no_perm_user_session.verify = VERIFY_SSL
no_perm_user_session.headers = HEADERS

In [7]:
def response_msg(message, response, is_json=True):
    """Append useful response details to provided message."""
    _body = response.text
    _detail = "<unknown>"
    if is_json:
        try:
            _body = response.json()
            _detail = _body.get("detail", _body.get("message", "<unknown>"))
        except json.JSONDecodeError:
            # ignore and revert to text body since it could not be parsed as JSON
            _body = response.text
    return "{} Response replied with ({}) [{}]\nContent: {}\n\n".format(message, response.status_code, _detail, _body)

In [14]:
CLEANUP_CALLED = False

def cleanup_test_data(skip_fail=True):
    """Cleanup test elements created during the test.

    Only raise at the complete end to attempt removal of as much element as possible in case of failures.
    """
    global CLEANUP_CALLED
    global TEST_DISABLE_CLEANUP
    if TEST_DISABLE_CLEANUP:
        print("WARNING - Skipping test data cleanup because of TEST_DISABLE_CLEANUP option!")
        return

    CLEANUP_CALLED = True
    failed = False

    # Cleanup Magpie users and resources
    magpie_items = [("users", "MAGPIE_TEST_USER", MAGPIE_TEST_USER),
                    ("users", "MAGPIE_NO_PERM_USER", MAGPIE_NO_PERM_USER)]

    # Find ids of resources to delete
    secured_geoserver_found = False
    resp = magpie_admin_session.get(f"{MAGPIE_URL}/services/{GEOSERVER_SERVICE}")
    if resp.status_code == 200:

        geoserver_res_id = resp.json()["service"]["resource_id"]
        _resp = magpie_admin_session.get(f"{MAGPIE_URL}/resources/{geoserver_res_id}")

        if _resp.status_code == 200:
            secured_geoserver_found = True
            _children = _resp.json()["resource"]["children"]
            for _, _child_info in _children.items():
                if _child_info["resource_name"] == workspace_name:
                    magpie_items.append(("resources", "test_workspace_resource", _child_info["resource_id"]))
    if not secured_geoserver_found:
        print(f"Failed to find the `{GEOSERVER_SERVICE}` service, which is expected to exist.")
        failed = True

    # Apply cleanup of Magpie items
    for item_type, item_name, item_value in magpie_items:
        try:
            _path = "{}/{}/{}".format(MAGPIE_URL, item_type, item_value)
            _resp = magpie_admin_session.delete(_path)
            _head = "Cleanup of {} :".format(item_name)
            if _resp.status_code == 200:
                print(_head, "OK")
            elif _resp.status_code == 404:
                print(_head, "WARNING - does not exist (maybe it failed to be created in the first place?)")
                failed = True  # most definitely the test is already failing at this point anyway
            else:
                print(_head, "ERROR - could not be removed")
                failed = True
        except Exception as exc:
            print("Magpie cleanup raised: [{}]".format(exc))
            failed = True

    # Cleanup Geoserver workspace (must remove each resource step by step)
    for path in [
        f"{GEOSERVER_REST_URL}/layers/{shapefile_name}",
        f"{GEOSERVER_REST_URL}/workspaces/{workspace_name}/datastores/{datastore_name}/featuretypes/{shapefile_name}",
        f"{GEOSERVER_REST_URL}/workspaces/{workspace_name}/datastores/{datastore_name}",
        f"{GEOSERVER_REST_URL}/workspaces/{workspace_name}"
    ]:
        try:
            _resp = geoserver_admin_session.delete(url=path)
            _head = "Cleanup of [{}]:".format(path)
            if _resp.status_code == 200:
                print(_head, "OK")
            elif _resp.status_code == 404:
                print(_head, "WARNING - does not exist (maybe it failed to be created in the first place?)")
                failed = True  # most definitely the test is already failing at this point anyway
            else:
                print(_head, "ERROR - could not be removed")
                failed = True
        except Exception as exc:
            print("Geoserver cleanup raised: [{}]".format(exc))
            failed = True

    if not skip_fail and failed:
        raise ValueError("Could not cleanup all test data.")

### Prepare Geoserver data

In [15]:
# create workspace
payload = {"workspace": {"name": workspace_name, "isolated": "True"}}
resp = geoserver_admin_session.post(url=f"{GEOSERVER_REST_URL}/workspaces", json=payload)

if resp.status_code != 201:
    cleanup_test_data()
    raise ValueError(response_msg("\nFailed to create Geoserver workspace `{}`.".format(workspace_name), resp))

In [18]:
# create datastore
payload = {
    "dataStore": {
        "name": datastore_name,
        "type": "Directory of spatial files (shapefiles)",
        "connectionParameters": {
            "entry": [
                {"$": "UTF-8",
                 "@key": "charset"},
                {"$": "shapefile",
                 "@key": "filetype"},
                {"$": "true",
                 "@key": "create spatial index"},
                {"$": "true",
                 "@key": "memory mapped buffer"},
                {"$": "GMT",
                 "@key": "timezone"},
                {"$": "true",
                 "@key": "enable spatial index"},
                {"$": f"http://{datastore_name}",
                 "@key": "namespace"},
                {"$": "true",
                 "@key": "cache and reuse memory maps"},
                {"$": f"file://{geoserver_datastore_path}",
                 "@key": "url"},
                {"$": "shape",
                 "@key": "fstype"},
            ]
        },
    }
}
resp = geoserver_admin_session.post(url=f"{GEOSERVER_REST_URL}/workspaces/{workspace_name}/datastores", json=payload)

if resp.status_code != 201:
    cleanup_test_data()
    raise ValueError(response_msg("\nFailed to create Geoserver datastore `{}`.".format(datastore_name), resp))

In [19]:
# publish shapefile
shapefile_payload = {
    "featureType": {
        "name": shapefile_name,
        "nativeCRS": """
                        GEOGCS[
                            "WGS 84",
                            DATUM[
                                "World Geodetic System 1984",
                                SPHEROID["WGS 84", 6378137.0, 298.257223563, AUTHORITY["EPSG","7030"]],
                                AUTHORITY["EPSG","6326"]
                            ],
                            PRIMEM["Greenwich", 0.0, AUTHORITY["EPSG","8901"]],
                            UNIT["degree", 0.017453292519943295],
                            AXIS["Geodetic longitude", EAST],
                            AXIS["Geodetic latitude", NORTH],
                            AUTHORITY["EPSG","4326"]
                        ]
                    """,
        "srs": "EPSG:4326",
        "projectionPolicy": "REPROJECT_TO_DECLARED",
        "maxFeatures": 5000,
        "numDecimals": 6,
    }
}

resp = geoserver_admin_session.post(
    url=f"{GEOSERVER_REST_URL}/workspaces/{workspace_name}/datastores/{datastore_name}/featuretypes",
    json=shapefile_payload
)

if resp.status_code != 201:
    cleanup_test_data()
    raise ValueError(response_msg("\nFailed to publish shapefile `{}` to Geoserver.".format(shapefile_name), resp))

### Prepare Magpie users and resources

In [20]:
def create_magpie_user(user_name, password, session):
    resp = magpie_admin_session.post(
        url=f"{MAGPIE_URL}/users",
        json={
            "user_name": user_name,
            "email": f"{user_name}@user.com",
            "password": password,
            "group_name": "users"},
        allow_redirects=False)
    if resp.status_code != 201:
        cleanup_test_data()
        raise ValueError(response_msg("\nCould not create test user [{}]".format(user_name), resp))
    session.cookies = magpie_signin(user_name, password).cookies

create_magpie_user(MAGPIE_TEST_USER, MAGPIE_TEST_PASSWORD, test_user_session)
create_magpie_user(MAGPIE_NO_PERM_USER, MAGPIE_NO_PERM_PASSWORD, no_perm_user_session)

In [21]:
# create/get resources
resp = magpie_admin_session.get(f"{MAGPIE_URL}/services/{GEOSERVER_SERVICE}")
if resp.status_code != 200:
    cleanup_test_data()
    raise ValueError(response_msg("\nCould not fetch secured Geoserver service details [{}].".format(GEOSERVER_SERVICE), resp))
geoserver_res_id = resp.json()["service"]["resource_id"]

def create_or_get_resource(resource_name, resource_type, parent_id):
    _path = MAGPIE_URL + "/resources"
    _data = {"parent_id": parent_id, "resource_type": resource_type, "resource_name": resource_name}
    _resp = magpie_admin_session.post(_path, json=_data)
    _msg = "[name: {}, type: {}, parent: {}].".format(resource_name, resource_type, parent_id)
    if _resp.status_code == 409:
        _path = "{}/{}".format(_path, parent_id)
        _resp = magpie_admin_session.get(_path)
        if _resp.status_code != 200:
            cleanup_test_data()
            raise ValueError(response_msg("Could not retrieve resource expected to exist {}.".format(_msg), _resp))
        _resource = _resp.json()["resource"]
        _children = _resource["children"]
        for _, _child_info in _children.items():
            if _child_info["resource_name"] == resource_name:
                return _child_info
    elif _resp.status_code != 201:
        cleanup_test_data()
        raise ValueError(response_msg("\nCould not create resource {}.".format(_msg), _resp))
    else:
        return _resp.json()["resource"]
    cleanup_test_data()
    raise ValueError(response_msg("\nCould not create or retrieve resource {}.".format(_msg), _resp))

workspace_res = create_or_get_resource(workspace_name, "workspace", geoserver_res_id)
layer_res = create_or_get_resource(shapefile_name, "layer", workspace_res["resource_id"])

In [22]:
def set_permission(permission, resource_id, resource_name, target_type, target_name):
    _path = "{}/{}/{}/resources/{}/permissions".format(MAGPIE_URL, target_type, target_name, resource_id)
    _data = {"permission": permission}
    _resp = magpie_admin_session.put(_path, json=_data)
    if _resp.status_code not in [200, 201]:
        _msg = "\nCleanup called before? {} (if called, following error could be expected)".format(CLEANUP_CALLED) + \
               "\nCould not set permission [{}] for [{}, {}] over resource [{}, {}]" \
               .format(permission, target_type, target_name, resource_name, resource_id)
        cleanup_test_data()
        raise ValueError(response_msg(_msg, _resp))

### Test user access

In [23]:
def has_access(path, session, user_name, expected_access):
    _resp = session.get(path, headers = {"Content-Type": None, "Cache-Control": "no-cache"})
    _code = _resp.status_code
    if _code in [200, 401, 403]:
        is_allowed = _code == 200
        str_allowed = "Allowed" if is_allowed else "Denied"
        print("Detail:\n"
              "  Resource: [{}]\n"
              "  User:     [{}]\n"
              "  Code:     [{}]\n"
              "  Access:   [{}]".format(path, user_name, _code, str_allowed))
        if is_allowed != expected_access:
            _msg = "Unexpected access to resource [{}]".format(path)
            raise ValueError(response_msg(_msg, _resp, is_json=False))
        return
    cleanup_test_data()
    print("Detail:\n"
          "  Resource: [{}]\n"
          "  User:     [{}]\n"
          "  Code:     [{}]\n"
          "  Access:   [Error]".format(path, user_name, _code))
    _msg = "Unexpected status code during access to resource [{}]".format(path)
    raise ValueError(response_msg(_msg, _resp, is_json=False))

In [24]:
# Prepare permissions, make sure permissions are denied for the test users before testing
set_permission("getfeature-deny-recursive", geoserver_res_id, GEOSERVER_SERVICE, "users", MAGPIE_TEST_USER)
set_permission("getfeature-deny-recursive", geoserver_res_id, GEOSERVER_SERVICE, "users", MAGPIE_NO_PERM_USER)
set_permission("describestoredqueries-deny-recursive", geoserver_res_id, GEOSERVER_SERVICE, "users", MAGPIE_TEST_USER)
set_permission("describestoredqueries-deny-recursive", geoserver_res_id, GEOSERVER_SERVICE, "users", MAGPIE_NO_PERM_USER)

In [25]:
# Test on workspace resource
res_path = f"{GEOSERVER_URL}/{workspace_name}/wfs?typeNames={workspace_name}:{shapefile_name}&request=GetFeature"

has_access(res_path, test_user_session, MAGPIE_TEST_USER, False)
has_access(res_path, no_perm_user_session, MAGPIE_NO_PERM_USER, False)

set_permission("getfeature-allow-recursive", workspace_res["resource_id"], workspace_res["resource_name"], "users", MAGPIE_TEST_USER)

has_access(res_path, test_user_session, MAGPIE_TEST_USER, True)
has_access(res_path, no_perm_user_session, MAGPIE_NO_PERM_USER, False)

Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=GetFeature]
  User:     [test-user-0e79b392-ce91-429c-a66c-e6242a3d7870]
  Code:     [403]
  Access:   [Denied]
Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=GetFeature]
  User:     [test-user-no-perm-c0d7a4f1-8f42-484c-935a-c01f6e6b1a02]
  Code:     [403]
  Access:   [Denied]
Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=GetFeature]
  User:     [test-user-0e79b392-ce91-429c-a66c-e6242a3d7870]
  Code:     [200]
  Access:   [Allowed]
Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=GetFeature]
  User:     [test-user-no-perm-c0d7a4f1-8f42-484c-935a-c01f6e6b1a02]
  Code:     [403]
  Access:   [Denied]


In [26]:
# Test on layer resource
res_path = f"{GEOSERVER_URL}/{workspace_name}/wfs?typeNames={workspace_name}:{shapefile_name}&request=DescribeStoredQueries"

has_access(res_path, test_user_session, MAGPIE_TEST_USER, False)
has_access(res_path, no_perm_user_session, MAGPIE_NO_PERM_USER, False)

set_permission("describestoredqueries-allow-match", layer_res["resource_id"], layer_res["resource_name"], "users", MAGPIE_TEST_USER)

has_access(res_path, test_user_session, MAGPIE_TEST_USER, True)
has_access(res_path, no_perm_user_session, MAGPIE_NO_PERM_USER, False)

Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=DescribeStoredQueries]
  User:     [test-user-0e79b392-ce91-429c-a66c-e6242a3d7870]
  Code:     [403]
  Access:   [Denied]
Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=DescribeStoredQueries]
  User:     [test-user-no-perm-c0d7a4f1-8f42-484c-935a-c01f6e6b1a02]
  Code:     [403]
  Access:   [Denied]
Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=DescribeStoredQueries]
  User:     [test-user-0e79b392-ce91-429c-a66c-e6242a3d7870]
  Code:     [200]
  Access:   [Allowed]
Detail:
  Resource: [https://pavics.ouranos.ca/geoserver/test_workspace/wfs?typeNames=test_workspace:Espace_Vert&request=DescribeStoredQueries]
  User:     [test-user-no-perm-c0d7a4f1-8f42-484c-935a-c01f6e6b1a02]
  Code:     [403]
  Access:   [Denied]


In [17]:
# final cleanup, everything is expected to have worked up to here,
# so force failure to ensure we return to original 'clean' state
cleanup_test_data(skip_fail=False)
print("All tests: OK")

Cleanup of MAGPIE_TEST_USER : OK
Cleanup of MAGPIE_NO_PERM_USER : OK
Cleanup of test_workspace_resource : OK
Cleanup of [https://pavics.ouranos.ca/geoserver/rest/layers/Espace_Vert]: OK
Cleanup of [https://pavics.ouranos.ca/geoserver/rest/workspaces/test_workspace/datastores/test_datastore/featuretypes/Espace_Vert]: OK
Cleanup of [https://pavics.ouranos.ca/geoserver/rest/workspaces/test_workspace/datastores/test_datastore]: OK
Cleanup of [https://pavics.ouranos.ca/geoserver/rest/workspaces/test_workspace]: OK
All tests: OK
